# Gemma 4 31B — Colab synthetic capacity smoke

**Purpose:** test whether `google/gemma-4-31B-it-qat-w4a16-ct` starts under a Colab GPU and whether the **frozen** plain/scalar-tool/array-tool parser probes work.

- **No benchmark tasks, reference patches, hidden tests, official scorer, or Kaggle submission.** Does not change frozen candidate or preregistered denominators.
- **Not an equivalent Kaggle A/B:** Colab uses one GPU (TP=1) and pip-installed vLLM; the Kaggle probe uses four L4 GPUs and the official scorer wheelhouse.
- **Do not run on T4 16 GB.** The preflight stops before installing/downloading unless a **single Ampere-or-newer GPU has at least 32 GiB free** and the local disk has at least 30 GiB free. This is only a conservative heuristic, not a guarantee that 32K context starts.
- Requires network access for vLLM/model download, and optionally a Colab secret named `HF_TOKEN`. Never paste secrets into notebook cells.
- Choose **Runtime → Change runtime type → GPU**, ideally **A100 40/80 GB or H100**; run cells sequentially. Closing Colab stops the temporary runtime.
- Local output `/content/gemma4_colab_31b_smoke_result.json` contains only scalar metrics; GPU logs remain private. If the model fails to start, classify it as **Colab runtime feasibility**, not model-quality evidence.

Frozen source: [capacity_31b_startup_v0.py](https://github.com/kolihhan/gemma-4-kaggle/blob/b13087b402ceef0aa0aa2ce55d1f7aa2c99108c3/kaggle/capacity_31b_startup_v0.py). References: [Google Gemma 4 model](https://huggingface.co/google/gemma-4-31B-it-qat-w4a16-ct), [vLLM Gemma 4 guide](https://github.com/vllm-project/recipes/blob/main/Google/Gemma4.md), [Colab resource policy](https://research.google.com/colaboratory/faq.html).

**Caution:** A successful Colab smoke does not validate Kaggle's submission runtime; it only removes some uncertainty about model loading and parser behavior.

In [ ]:
# Resource gate: does not install packages, download weights, or use the GPU.
import json, os, pathlib, shutil, subprocess, time
MODEL_ID = "google/gemma-4-31B-it-qat-w4a16-ct"
MAX_MODEL_LEN = 32768
GPU_MEMORY_UTILIZATION = 0.90
MIN_FREE_GPU_GIB = 32.0  # conservative screening heuristic, not a guaranteed vLLM fit
MIN_FREE_DISK_GIB = 30.0
RESULT_PATH = pathlib.Path("/content/gemma4_colab_31b_smoke_result.json")
SOURCE_COMMIT = "b13087b402ceef0aa0aa2ce55d1f7aa2c99108c3"
SOURCE_BLOB_SHA1 = "c458c21c04e0b456cb3b0e01f6f145129445524c"

gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free,compute_cap",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=False,
)
gpus = []
if gpu_query.returncode == 0:
    for line in gpu_query.stdout.splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) != 5:
            continue
        try:
            gpus.append({
                "index": int(parts[0]), "name": parts[1],
                "total_gib": round(float(parts[2]) / 1024, 2),
                "free_gib": round(float(parts[3]) / 1024, 2),
                "compute_capability": float(parts[4]),
            })
        except ValueError:
            continue

disk_free_gib = round(shutil.disk_usage("/content").free / (1024 ** 3), 2)
eligible = (
    len(gpus) == 1
    and gpus[0]["free_gib"] >= MIN_FREE_GPU_GIB
    and gpus[0]["compute_capability"] >= 8.0
    and disk_free_gib >= MIN_FREE_DISK_GIB
)
preflight = {
    "gpu_inventory": gpus,
    "disk_free_gib": disk_free_gib,
    "requires_one_gpu": True,
    "min_free_gpu_gib": MIN_FREE_GPU_GIB,
    "min_free_disk_gib": MIN_FREE_DISK_GIB,
    "eligible_for_colab_smoke": eligible,
    "experiment_only": "synthetic startup/tool parsing; not the official Kaggle scorer",
    "competition_tasks_used": 0,
}
print(json.dumps(preflight, indent=2, sort_keys=True))
if not eligible:
    RESULT_PATH.write_text(json.dumps({
        "status": "BLOCKED_BY_COLAB_PREFLIGHT", "preflight": preflight,
        "competition_tasks_used": 0,
    }, indent=2) + "\n", encoding="utf-8")
    raise RuntimeError(
        "Colab GPU or storage does not meet the conservative 31B probe gate. "
        "Do not download the model. Try an A100 40/80 GB or another GPU with "
        "at least 32 GiB free, and >=30 GiB free disk."
    )

In [ ]:
# Runs only after the GPU/disk preflight passed.
# vLLM is installed into this disposable Colab runtime, NOT into the repo.
import importlib.metadata, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "vllm"], check=True)
print("vllm_version =", importlib.metadata.version("vllm"))
print("If vLLM reports a Torch/CUDA ABI error, restart the Colab runtime and run from cell 1.")

In [ ]:
# Reuse the frozen, synthetic-only repository probes.
# We verify the source's Git blob SHA1 before executing it. No task data is downloaded.
import hashlib, urllib.request
source_url = (
    "https://raw.githubusercontent.com/kolihhan/gemma-4-kaggle/"
    + SOURCE_COMMIT + "/kaggle/capacity_31b_startup_v0.py"
)
with urllib.request.urlopen(source_url, timeout=30) as response:
    source_bytes = response.read()
git_blob = hashlib.sha1(
    b"blob " + str(len(source_bytes)).encode("ascii") + b"\0" + source_bytes
).hexdigest()
assert git_blob == SOURCE_BLOB_SHA1, "Frozen probe code does not match Git blob"
probe_module = {"__name__": "colab_frozen_probe", "__file__": "capacity_31b_startup_v0.py"}
exec(compile(source_bytes, "capacity_31b_startup_v0.py", "exec"), probe_module)
for required in ("probe_plain", "probe_scalar_tool", "probe_array_tool", "http_json"):
    assert callable(probe_module[required]), required
print("FROZEN_REPOSITORY_PROBES=VERIFIED", SOURCE_COMMIT[:12])
print("COMPETITION_TASKS_LOADED=0")

In [ ]:
# Start the exact Gemma 4 31B QAT model and run three SYNTHETIC-only probes.
# No Kaggle datasets, no official scorer, no benchmark tasks, no public upload.
import hashlib, importlib.metadata, json, os, pathlib, signal, subprocess, time
from urllib.error import HTTPError, URLError
LOG_PATH = pathlib.Path("/content/gemma4_colab_vllm_private.log")
BASE_URL = "http://127.0.0.1:8000"
STARTUP_TIMEOUT_SECONDS = 1800
proc = None
log_handle = None
started = time.monotonic()
summary = {
    "version": "capacity-v0-colab-31b-synthetic-smoke",
    "status": "started",
    "model": MODEL_ID,
    "model_len": MAX_MODEL_LEN,
    "tensor_parallel_size": 1,  # Colab single GPU, official Kaggle probe used TP=4
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
    "colab_gpu": preflight["gpu_inventory"],
    "vllm_version": importlib.metadata.version("vllm"),
    "source_commit": SOURCE_COMMIT,
    "source_git_blob_sha1": SOURCE_BLOB_SHA1,
    "competition_tasks_used": 0,
    "official_kaggle_scorer": False,
    "plain": None,
    "scalar": None,
    "array": None,
}
try:
    env = os.environ.copy()
    env.update({
        "HF_HUB_DISABLE_TELEMETRY": "1",
        "TRANSFORMERS_NO_TF": "1",
        "TRANSFORMERS_NO_FLAX": "1",
        "VLLM_NO_USAGE_STATS": "1",
        "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
    })
    # Optional Colab Secret: HF_TOKEN (never print or paste it in a notebook).
    try:
        from google.colab import userdata
        secret = userdata.get("HF_TOKEN")
        if secret:
            env["HF_TOKEN"] = secret
    except Exception:
        pass

    cmd = [
        "vllm", "serve", MODEL_ID,
        "--host", "127.0.0.1", "--port", "8000",
        "--tensor-parallel-size", "1",
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--dtype", "bfloat16",
        "--enable-auto-tool-choice",
        "--tool-call-parser", "gemma4",
        "--reasoning-parser", "gemma4",
    ]
    log_handle = LOG_PATH.open("w", encoding="utf-8")
    proc = subprocess.Popen(
        cmd, stdout=log_handle, stderr=subprocess.STDOUT,
        env=env, start_new_session=True,
    )
    model_api_id = None
    while time.monotonic() - started < STARTUP_TIMEOUT_SECONDS:
        if proc.poll() is not None:
            raise RuntimeError("vLLM process exited before becoming ready")
        try:
            models = probe_module["http_json"]("GET", "/v1/models").get("data") or []
            if models:
                model_api_id = str(models[0]["id"])
                break
        except (HTTPError, URLError, ConnectionError, TimeoutError, OSError):
            pass
        time.sleep(10)
    if model_api_id is None:
        raise RuntimeError("vLLM did not become ready within the startup bound")

    summary["startup_seconds"] = round(time.monotonic() - started, 3)
    summary["status"] = "server_ready"
    summary["plain"] = probe_module["probe_plain"](model_api_id)
    summary["scalar"] = probe_module["probe_scalar_tool"](model_api_id)
    summary["array"] = probe_module["probe_array_tool"](model_api_id)
    summary["status"] = "smoke_completed"
    summary["probe_gate_pass"] = (
        summary["plain"].get("http_ok") is True
        and summary["scalar"].get("recognized_tool_call") is True
        and summary["scalar"].get("arguments_match_fixture") is True
        and summary["array"].get("recognized_tool_call") is True
        and summary["array"].get("arguments_match_fixture") is True
    )
except Exception as exc:
    summary["status"] = "error"
    summary["error_type"] = type(exc).__name__
    summary["error_digest_sha256"] = hashlib.sha256(str(exc).encode("utf-8")).hexdigest()
finally:
    if proc is not None and proc.poll() is None:
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
            proc.wait(timeout=25)
        except Exception:
            try:
                os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
            except Exception:
                pass
    if log_handle is not None:
        log_handle.close()
    summary["total_seconds"] = round(time.monotonic() - started, 3)
    RESULT_PATH.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(json.dumps(summary, indent=2, sort_keys=True))
    print("Local result:", RESULT_PATH)
    print("Private startup diagnostics:", LOG_PATH)
    print("THIS IS NOT A KAGGLE RESOLVED-TASK RESULT.")

In [ ]:
# Optional: download the metrics-only JSON to share for analysis.
# Do NOT upload the private vLLM log unless you review it first.
from google.colab import files
files.download(str(RESULT_PATH))